# Week 3a.2: Structured Output

Most real-word agent responses are used to feed data into a database, a UI, or another agent. Raw text output isn't suitable for this purpose.

Instead, structured output is used to enable the model to retun data in a predefined format. We'll look at one such method.

In [8]:
from dotenv import load_dotenv
import os
import logging

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."

# silence a noisy advisory warning from the Google SDK
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

print("API key loaded")

API key loaded


## 0. Model Setup

Execute one of the following to define the model.

If using Ollama, you will need to start it first (simply open the chat UI and send a message):
- https://docs.langchain.com/oss/python/integrations/chat/ollama

Verify that Ollama is serving a model:
- http://localhost:11434/

In [9]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", reasoning=False)

In [10]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

## 1. The problem

In [12]:
email = """Subject: STILL waiting

I ordered a mechanical keyboard two weeks ago, order HT-1002, and it has still
not arrived. This is the third time I am writing. If it is not here by Friday
I want my money back."""

response = model.invoke(f"Extract the order id and the problem from this email:\n\n{email}")
print(response.text)

* **Order ID:** HT-1002
* **Problem:** The order has not arrived after two weeks (delayed delivery, lack of communication/response).


A perfectly good answer for a human, but try writing code against it! 

The phrasing changes run to run; string parsing breaks with it. We need to tell the model the exact shape we want.

## 2. Describing the shape: Pydantic

We describe the shape as a **Pydantic model**: a **class** listing each field with a **type** and a **description**. 

The descriptions are not comments. Like tool docstrings, the model reads them to decide what goes where. 

`Literal` restricts a field to fixed choices.

### Understanding `BaseModel`
The core Pydantic class used to define data structures, enforce type hints at runtime, and validate incoming data

The class needs to inherit from BaseModel
```python    
    class YourClassName(BaseModel):
```


In [14]:
from pydantic import BaseModel, Field
from typing import Literal

class User(BaseModel):
    username: str = Field(description = "The username used to log in to the system")
    tier: Literal["free", "paid"] = Field(description = "Account tier")

Let's define a SupportTicket class to get a structured record of customer support requests:
- `order_id`: a string with description = The order id mentioned, e.g. HT-1001
- `category`: Literal with possibilities billing, shipping, returns, other. Description: what kind of problem this is
- `summary`: a string with description = The problem in one sentence
- `urgent`: a bool with description = whether the customer needs an immediate response

In [15]:
from pydantic import BaseModel, Field
from typing import Literal

class SupportTicket(BaseModel):
    """A structured record of one customer support request."""
    order_id: str = Field(description="The order id mentioned, e.g. HT-1001")
    category: Literal["billing", "shipping", "returns", "other"] = Field(
        description="What kind of problem this is")
    summary: str = Field(description="The problem in one sentence")
    urgent: bool = Field(description="Whether the customer needs an immediate response")

## 3. Structured output from models

`with_structured_output` wraps the model so it must answer in that shape. 

Under the hood this is tool calling: the schema is handed to the model as the one tool it has to call.

- https://docs.langchain.com/oss/python/langchain/structured-output

In [16]:
structured_model = model.with_structured_output(SupportTicket)

ticket = structured_model.invoke(f"Extract a support ticket from this email:\n\n{email}")
ticket

SupportTicket(order_id='HT-1002', category='shipping', summary='Customer is waiting for an order that has not arrived after two weeks and is threatening to request a refund if it does not arrive by Friday.', urgent=True)

In [17]:
print(ticket.order_id)
print(ticket.category)
print(ticket.urgent)
print(type(ticket))

HT-1002
shipping
True
<class '__main__.SupportTicket'>


Returns an actual Python object. 

`ticket.order_id` is a `str`, `ticket.urgent` is a `bool`, and `category` is guaranteed to be one of the four allowed values.

In [18]:
#EXERCISE: define a ProductReview model and extract one from the text below.
# Fields: rating (int, 1 to 5), pros (list[str]), cons (list[str]).
# Give every field a description.

review = """These headphones sound amazing for the price and the battery lasts
all week. The ear cushions get sweaty after an hour though, and the app is
useless. Solid 4 out of 5 from me."""


## 4. Structured output from agents

To get structured data from agents, we'll need to add `response_format` to `create_agent`. 
- `response_format=SupportTicket`

With this format requirement, the final answer will arrive twice.
- The prose reply in `messages`, and the typed object under `structured_response`. Tools still work along the way.

In [19]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="You turn customer emails into support tickets.",
    response_format=SupportTicket
    # TODO
)

result = agent.invoke({"messages": [HumanMessage(content=email)]})
result["structured_response"]

SupportTicket(order_id='HT-1002', category='shipping', summary='Customer is experiencing severe delivery delays and is requesting a refund if the order does not arrive by Friday.', urgent=True)

In [20]:
result

{'messages': [HumanMessage(content='Subject: STILL waiting\n\nI ordered a mechanical keyboard two weeks ago, order HT-1002, and it has still\nnot arrived. This is the third time I am writing. If it is not here by Friday\nI want my money back.', additional_kwargs={}, response_metadata={}, id='ab2f8906-e183-419e-ad35-e581bf9789e2'),
  AIMessage(content=[{'type': 'text', 'text': '{\n  "order_id": "HT-1002",\n  "category": "shipping",\n  "summary": "Customer is experiencing severe delivery delays and is requesting a refund if the order does not arrive by Friday.",\n  "urgent": true\n}', 'extras': {'signature': 'EmAKXgFpFH0TB4d3oqx3YDBd8DsuE8PfDxLP4HKPwjjwItE8OfGNId03U/mBSI0EaOObjYTtYRzRp3U0EN4wsVwCDvotiXnY1u6TZ2cwO1l+GQbsd+S40/rotamNPwHu+AI='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0d98b-71bd-7ff3-a4dd-fca80f18d6e6-0', tool_calls=[], invalid_tool_call

In [21]:
result["messages"][1].tool_calls

[]